# FabricDaxLoadTest — Queries

Manages the DAX query corpus stored at `Files/queries.json` in the
**`LoadTests`** lakehouse (same workspace folder as this notebook).

The `Run.ipynb` notebook reads this file by default. Inline overrides
on `Run.ipynb` (`QUERIES_INLINE`) take precedence when set.

`queries.json` is either:

- a JSON list of strings: `["EVALUATE ROW(\"x\", 1)", ...]`, or
- a JSON list of objects with a `query` key:
  `[{"query": "EVALUATE ...", "name": "...", "tags": [...]}]` — extra
  fields are preserved on round-trip but ignored by the runner.

In [ ]:
# ── 1. Locate the LoadTests lakehouse ─────────────────────────────────────────
import json, notebookutils
ctx = notebookutils.runtime.context
WS_ID = ctx["currentWorkspaceId"]
LH_ABFSS = f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com/LoadTests.Lakehouse"
QPATH = f"{LH_ABFSS}/Files/queries.json"
print(f"Lakehouse: {LH_ABFSS}")
print(f"Catalog  : {QPATH}")

In [ ]:
# ── 2. Read the current catalog ───────────────────────────────────────────────
try:
    raw = notebookutils.fs.head(QPATH, 1024 * 1024 * 4)
    queries = json.loads(raw)
    print(f"Loaded {len(queries)} queries from {QPATH}")
except Exception as e:
    print(f"(no existing catalog — will create on save: {e})")
    queries = []

import pandas as pd
def to_df(qs):
    rows = []
    for i, q in enumerate(qs):
        if isinstance(q, str):
            rows.append({"i": i, "name": "", "tags": "", "query": q})
        else:
            rows.append({
                "i":     i,
                "name":  q.get("name", ""),
                "tags":  ",".join(q.get("tags", [])) if isinstance(q.get("tags"), list) else (q.get("tags") or ""),
                "query": q.get("query", ""),
            })
    return pd.DataFrame(rows)

df = to_df(queries)
display(df.head(50))

In [ ]:
# ── 3. Edit the catalog ───────────────────────────────────────────────────────
# Edit the Python literal below and re-run cells 3 + 4. Each entry is a string
# (just the DAX) or a dict with optional name/tags. Strings work for simple
# cases; switch to dicts when you want labels in the report.
new_queries = [
    # Replace these examples with your real corpus.
    "EVALUATE ROW(\"x\", 1)",
    {"name": "topn-sales", "tags": ["sales", "topn"],
     "query": "EVALUATE TOPN(10, SUMMARIZECOLUMNS('Date'[Year], \"Sales\", [Sales Amount]))"},
]
print(f"Prepared {len(new_queries)} queries.")
display(to_df(new_queries))

In [ ]:
# ── 4. Save back to OneLake ───────────────────────────────────────────────────
import os, tempfile
tmp = tempfile.NamedTemporaryFile("w", suffix=".json", delete=False, encoding="utf-8")
json.dump(new_queries, tmp, indent=2)
tmp.close()
notebookutils.fs.cp(f"file://{tmp.name}", QPATH, overwrite=True)
os.unlink(tmp.name)
print(f"Saved {len(new_queries)} queries to {QPATH}")